# Budowa kostki podsumowującej rozliczenia telekomunikacyjne (asekuracja przychodu) za pomocą PROC SUMMARY

## Streszczenie

Zespół ds. asekuracji przychodu w operatorze bezprzewodowym wstępnie agreguje miesiąc rekordów rozliczeniowych abonent-dzień do kompaktowej kostki podsumowującej, aby analitycy mogli zagłębiać się w rozliczony przychód według regionu, taryfy i typu połączenia bez ponownego skanowania surowej tabeli faktów. `PROC SUMMARY` zwija 100 rekordów abonent-dzień w wielowymiarowy zbiór komórek: najdrobniejsza ziarnistość (region x taryfa x typ połączenia) wypełnia 25 z 27 możliwych komórek, natomiast nazwane podkostki dają marginalia, o które analitycy pytają najczęściej. W tym przykładowym miesiącu operator rozliczył \$222.78 w trzech aktywnych regionach, przy czym Południe (\$97.44) i Wschód (\$86.94) łącznie odpowiadają za 83% przychodu, a Północ zamyka stawkę z \$38.40. Najbogatszą pojedynczą podkostką jest usługa Głos w taryfie Plus (\$59.06 na 18 rekordach), a ranking komórek według wydajności na rekord ujawnia komórki zużycia danych jako najbardziej wartościowe cele audytu przecieków (najwyższa wydajność \$7.87/rekord). Każda liczba poniżej pochodzi bezpośrednio z wykonanego kodu.

## Źródła danych

| Zbiór danych | Wiersze | Ziarnistość | Kluczowe zmienne |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Jeden wiersz na podsumowanie użycia abonent-dzień | `region` (Wschód/Południe/Północ), `plan_tier` (Przedpłata/Podstawowy/Plus), `call_type` (Głos/SMS/Dane), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Jeden wiersz na niepustą komórkę (region x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Jeden wiersz na komórkę nazwanych podkostek zagłębienia | `_TYPE_`, `_FREQ_`, `rev_total` |

Wszystkie dane są generowane wewnątrz notatnika przy pomocy `call streaminit()` / `rand()`; nie są wymagane pliki zewnętrzne ani dostęp do sieci. To środowisko działa bez licencji, więc zapisywane zbiory danych są ograniczone do 100 obserwacji - scenariusz jest tak dobrany, aby kostka była w pełni wypełniona w ramach tego limitu.

## Od surowych rekordów szczegółów połączeń do przeszukiwalnej kostki

Operatorzy bezprzewodowi rozliczają przychód z milionów codziennych zdarzeń użycia. Aby analitycy ds. asekuracji przychodu mogli odpowiadać na pytania w rodzaju *"Jaki był przychód z usługi głosowej w taryfie Plus w regionie Południe w zeszłym miesiącu?"* bez ponownego skanowania surowej tabeli faktów za każdym razem, **wstępnie agregujemy** dane do kompaktowej kostki podsumowującej.

`PROC SUMMARY` to koń roboczy Base SAS dokładnie do tego celu: grupuje płaską tabelę faktów według jednego lub więcej wymiarów `CLASS` i zapisuje żądane statystyki do zbioru wyjściowego, tagując każdy wiersz `_TYPE_` (które wymiary są aktywne) i `_FREQ_` (rekordy stojące za komórką). Ten zbiór wyjściowy *jest* kostką wielowymiarową - tą samą strukturą zestawień, jaką udostępniałoby narzędzie OLAP, zmaterializowaną jako zwykły zbiór danych SAS, który można drukować, łączyć i wycinać.

Ten notatnik:

1. Generuje realistyczny miesiąc rekordów rozliczeniowych abonent-dzień.
2. Buduje kostkę o najdrobniejszej ziarnistości (region x taryfa x typ połączenia) za pomocą `PROC SUMMARY NWAY`.
3. Materializuje nazwane **podkostki zagłębienia** za pomocą instrukcji `TYPES`.
4. Rzutuje przychód na bazę abonentów za pomocą `WEIGHT`, zagłębia się w jeden region i rankinguje komórki według wydajności na rekord do triażu przecieków.

## Krok 1 - Generowanie syntetycznych danych szczegółów połączeń / rozliczeń

Każdy wiersz podsumowuje użycie jednej usługi przez jednego abonenta w jednym dniu. Używamy `call streaminit` dla powtarzalności i `rand()` do losowania realistycznych rozkładów: przychód i użycie skalują się z taryfą, przychód głosowy śledzi rozliczane minuty, a przychód z danych śledzi megabajty. Każde `RAND('table', ...)` ma jedno prawdopodobieństwo na kategorię, tak aby każdy region, taryfa i typ połączenia wystąpiły w próbie 100 rekordów. Dołączona jest niewielka waga ankietowa `subscriber_wt`, aby móc później zademonstrować miarę ważoną.

In [1]:
DANE work.cdr_billing;
    CALL streaminit(20260131);
    DŁUGOŚĆ region $10 plan_tier $14 call_type $9 device_class $10 bill_month $7;
    ETYKIETA revenue       = "Rozliczony przychód (USD)"
          call_minutes  = "Rozliczane minuty połączeń głosowych"
          data_mb       = "Wolumen danych (MB)"
          subscriber_wt = "Waga ankietowa abonenta";

    POWTÓRZ i = 1 TO 100;
        /* --- Wymiary: jedno prawdopodobieństwo na poziom, suma do 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        WYBIERZ (r);
            GDY (1) region = "Wschód";
            GDY (2) region = "Południe";
            INACZEJ region = "Północ";
        KONIEC;

        p = rand("table", 0.30, 0.40, 0.30);
        WYBIERZ (p);
            GDY (1) plan_tier = "Przedpłata";
            GDY (2) plan_tier = "Podstawowy";
            INACZEJ plan_tier = "Plus";
        KONIEC;

        c = rand("table", 0.50, 0.30, 0.20);
        WYBIERZ (c);
            GDY (1) call_type = "Głos";
            GDY (2) call_type = "SMS";
            INACZEJ call_type = "Dane";
        KONIEC;

        JEŚLI rand("uniform") < 0.55 WTEDY device_class = "Smartfon";
        PRZECIWNIE device_class = "Zwykły";

        bill_month = "2026-01";

        /* --- Miary, skalowane taryfą i usługą --- */
        tier_mult = (plan_tier = "Przedpłata")*0.7
                  + (plan_tier = "Podstawowy")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Głos")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Dane")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        WYJŚCIE;
    KONIEC;
    USUŃ i r p c tier_mult base_rev;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.cdr_billing(obs=8) ETYKIETA noobs;
    TYTUŁ "Przykładowe rekordy rozliczeniowe abonent-dzień";
WYKONAJ;

                                    Przykładowe rekordy rozliczeniowe abonent-dzień                                     

   region    plan_tier  call_type  device_class  bill_month      Rozliczane minuty połączeń głosowych  Wolumen danych (MB)   Rozliczony przychód (USD)  Waga ankietowa abonenta
Północ     Plus         SMS        Smartfon      2026-01                                            0                    0                        0.67                     1.13
Południe   Przedpłata   SMS        Zwykły        2026-01                                            0                    0                        0.94                     1.42
Południe   Plus         SMS        Smartfon      2026-01                                            0                    0                        0.99                     0.86
Południe   Plus         SMS        Smartfon      2026-01                                            0                    0                        1.01                     1.0


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Krok 2 - Budowa kostki o najdrobniejszej ziarnistości za pomocą PROC SUMMARY NWAY

`NWAY` zachowuje tylko jedną, najbardziej szczegółową kombinację wszystkich wymiarów `CLASS` - tutaj każdą wypełnioną kombinację (region x plan_tier x call_type). Dla każdej komórki przechowujemy `SUM`, `MEAN` i `MAX` przychodu plus łączne minuty i megabajty, tak aby analityk mógł odczytać łączny przychód na komórkę, wyznaczyć średnią na rekord i wykryć największą pojedynczą opłatę. `_FREQ_` zapisuje, ile dni-abonent stoi za każdą komórką. Usuwamy tu `_TYPE_`, ponieważ na poziomie ziarnistości NWAY każdy wiersz ma ten sam typ.

In [2]:
PROCEDURA summary DANE=work.cdr_billing NWAY;
    KLASA region plan_tier call_type;
    ZMIENNA revenue call_minutes data_mb;
    WYJŚCIE out=work.cube_nway(USUŃ=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.cube_nway(obs=12) noobs ETYKIETA;
    TYTUŁ "Komórki kostki NWAY (region * taryfa * typ połączenia)";
    ETYKIETA region='Region' plan_tier='Taryfa' call_type='Typ połączenia'
          _freq_='Liczba rekordów' rev_total='Suma przychodu'
          rev_mean='Średni przychód' rev_max='Maks. przychód'
          min_total='Suma minut' data_total='Suma MB';
    format rev_total rev_mean rev_max min_total data_total comma10.2;
WYKONAJ;

                                 Komórki kostki NWAY (region * taryfa * typ połączenia)                                 

   Region       Taryfa    Typ połączenia   Liczba rekordów  Suma przychodu    Średni przychód   Maks. przychód  Suma minut   Suma MB
Południe   Plus         Dane                             2           11.92               5.96            10.16        0.00  1,122.90
Południe   Plus         Głos                             8           27.07               3.38             5.99      521.90      0.00
Południe   Plus         SMS                              5            5.16               1.03             1.35        0.00      0.00
Południe   Podstawowy   Dane                             3           12.02               4.01             5.98        0.00  1,368.50
Południe   Podstawowy   Głos                             9           22.51               2.50             4.76      451.80      0.00
Południe   Podstawowy   SMS                              2            2.01      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Krok 3 - Materializacja nazwanych podkostek zagłębienia za pomocą TYPES

Kostka OLAP wstępnie przechowuje zestawienia, po których analitycy nawigują najczęściej. Instrukcja `TYPES` robi dokładnie to: każdy termin prosi `PROC SUMMARY` o wyemitowanie jednej podkostki. Żądamy sumy całkowitej `()`, marginalium `region` oraz dwuwymiarowych podkostek `region*plan_tier` i `call_type*plan_tier` - ścieżek zagłębienia, jakie udostępniałby pulpit przychodowy. SAS oznacza każdą podkostkę kodem `_TYPE_` (maska bitowa nad listą `CLASS`), więc jeden zbiór wyjściowy niesie każdy poziom hierarchii.

In [3]:
PROCEDURA summary DANE=work.cdr_billing;
    KLASA region plan_tier call_type;
    ZMIENNA revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    WYJŚCIE out=work.cube_hier
           sum(revenue)=rev_total;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.cube_hier noobs ETYKIETA;
    TYTUŁ "Zestawienia podkostek: suma całkowita, region, region*taryfa, typ_połączenia*taryfa";
    ZMIENNA _type_ region plan_tier call_type _freq_ rev_total;
    ETYKIETA region='Region' plan_tier='Taryfa' call_type='Typ połączenia'
          _freq_='Liczba rekordów' rev_total='Suma przychodu';
    format rev_total comma10.2;
WYKONAJ;

                  Zestawienia podkostek: suma całkowita, region, region*taryfa, typ_połączenia*taryfa                   

_TYPE_     Region       Taryfa    Typ połączenia   Liczba rekordów  Suma przychodu
     0                                                         100          222.78
     3             Plus         Dane                             3           17.46
     3             Plus         Głos                            18           59.06
     3             Plus         SMS                             13           11.75
     3             Podstawowy   Dane                             8           38.06
     3             Podstawowy   Głos                            20           42.33
     3             Podstawowy   SMS                              8            8.03
     3             Przedpłata   Dane                             7           14.50
     3             Przedpłata   Głos                            16           24.77
     3             Przedpłata   SMS             


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Krok 4 - Rzutowanie ważone, zagłębienie regionalne i triaż przecieków

Trzy odczyty, które zespół ds. asekuracji przychodu faktycznie wykonuje na kostce:

- **Rzutowanie ważone.** Dołączenie `WEIGHT=subscriber_wt` do podsumowania `region*plan_tier` raportuje przychód przeskalowany do pełnej bazy abonentów, którą reprezentuje próba, a nie do surowych zebranych wierszy.
- **Zagłębienie.** Filtrowanie kostki NWAY do jednego regionu rozwija pojedynczą gałąź hierarchii - tutaj Południe - do jej szczegółu taryfa-po-usłudze.
- **Triaż przecieków.** Sortowanie komórek według średniego przychodu na rekord ujawnia komórki o najwyższej wydajności; komórki o niskiej częstości i wysokiej wydajności są dokładnie tym, co audyt bada pod kątem błędnie taryfikowanego lub przeciekającego przychodu.

In [4]:
/* Ważony przychód rzutowany na bazę abonentów */
PROCEDURA summary DANE=work.cdr_billing NWAY;
    KLASA region plan_tier;
    ZMIENNA revenue;
    WAGA subscriber_wt;
    WYJŚCIE out=work.cube_wt(USUŃ=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.cube_wt noobs ETYKIETA;
    TYTUŁ "Ważony przychód według region * taryfa (rzutowany na bazę abonentów)";
    ETYKIETA region='Region' plan_tier='Taryfa' rev_weighted='Ważony przychód'
          records='Liczba rekordów';
    format rev_weighted comma10.2;
WYKONAJ;

/* Zagłębienie w gałąź regionu Południe kostki */
PROCEDURA DRUKUJ DANE=work.cube_nway noobs ETYKIETA;
    GDZIE region = "Południe";
    TYTUŁ "Zagłębienie w Południe: komórki przychodu według taryfy i typu połączenia";
    ZMIENNA plan_tier call_type _freq_ rev_total rev_mean;
    ETYKIETA plan_tier='Taryfa' call_type='Typ połączenia'
          _freq_='Liczba rekordów' rev_total='Suma przychodu'
          rev_mean='Średni przychód';
    format rev_total rev_mean comma10.2;
WYKONAJ;

/* Ranking komórek według wydajności na rekord dla triażu przecieków */
PROCEDURA SORTUJ DANE=work.cube_nway out=work.cube_ranked;
    WEDŁUG MALEJĄCO rev_mean;
WYKONAJ;

PROCEDURA DRUKUJ DANE=work.cube_ranked(obs=6) noobs ETYKIETA;
    TYTUŁ "Komórki o najwyższym średnim przychodzie (wydajność na rekord)";
    ZMIENNA region plan_tier call_type _freq_ rev_mean rev_max;
    ETYKIETA region='Region' plan_tier='Taryfa' call_type='Typ połączenia'
          _freq_='Liczba rekordów' rev_mean='Średni przychód'
          rev_max='Maks. przychód';
    format rev_mean rev_max comma10.2;
WYKONAJ;

                          Ważony przychód według region * taryfa (rzutowany na bazę abonentów)                          

   Region       Taryfa    Ważony przychód   Liczba rekordów
Południe   Plus                     56.29                15
Południe   Podstawowy               58.63                14
Południe   Przedpłata               27.77                10
Północ     Plus                     22.42                 7
Północ     Podstawowy               18.30                 7
Północ     Przedpłata               20.56                 9
Wschód     Plus                     59.59                12
Wschód     Podstawowy               50.85                15
Wschód     Przedpłata               29.77                11

                       Zagłębienie w Południe: komórki przychodu według taryfy i typu połączenia                        

     Taryfa    Typ połączenia   Liczba rekordów  Suma przychodu    Średni przychód
Plus         Dane                             2           11.92         


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpretacja wyników

Kostka podsumowująca zamienia 100 surowych wierszy abonent-dzień w kompaktowy zbiór wstępnie zagregowanych komórek, które odpowiadają na pytania zagłębiające natychmiast, bez ponownego skanowania tabeli faktów:

- **Gdzie mieszka przychód.** Marginalium `region` (`_TYPE_=4`) pokazuje, że Południe rozliczyło \$97.44, a Wschód \$86.94 - razem 83% z \$222.78 miesiąca - podczas gdy Północ wniosła \$38.40. W podkostce `call_type*plan_tier` (`_TYPE_=3`) usługa Głos w taryfie Plus jest pojedynczą najbogatszą komórką z \$59.06 na 18 rekordach, z usługą Głos w taryfie Podstawowy zaraz za nią - \$42.33.
- **Rzutowanie ważone.** Zastosowanie wagi ankietowej przetasowuje ranking w stronę taryf, których abonenci niosą większą wagę: Wschód-Plus (\$59.59) i Południe-Podstawowy (\$58.63) prowadzą w rzutowanym przychodzie `region*plan_tier` - inny obraz niż niewazone sumy regionalne i przypomnienie, aby raportować przychód rzutowany, a nie próbkowany, przy szacowaniu ekspozycji.
- **Wydajność na rekord i triaż przecieków.** Ranking komórek NWAY według `rev_mean` stawia komórki użycia danych na szczycie - Północ-Podstawowy-Dane (\$7.87/rekord) i Południe-Plus-Dane (\$5.96/rekord) - potwierdzając, że intensywne użycie danych napędza najwyższy przychód na rekord. Liderzy o niskiej częstości (jeden lub dwa rekordy) są dokładnie tym, dla czego analityk ds. asekuracji przychodu wyciągnąłby leżące u podstaw rekordy szczegółów połączeń, aby potwierdzić, że wysoka opłata jest poprawnie taryfikowana, a nie jest błędem.

Dla zespołu ds. asekuracji przychodu ta kostka jest fundamentem pulpitów wariancji: porównaj rozliczony przychód z oczekiwanym przychodem z cennika na komórkę (region, taryfa, typ połączenia), a komórki, których średnia lub ważona suma odchylają się najbardziej, stają się przypadkami przecieku wartymi audytu. Ponieważ cała struktura jest zwykłym zbiorem danych SAS, kostkę następnego miesiąca można zsumować, zróżnicować lub połączyć z cennikiem tymi samymi narzędziami Base SAS.